In [0]:
from pyspark.sql.functions import col, current_timestamp

In [0]:
kafka_bootstrap = "eh-streaming-demo.servicebus.windows.net:9093"
eh_conn_str = dbutils.secrets.get("kv-scope", "eventhub-conn-string")

sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{eh_conn_str}";'
)

In [0]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", "wiki-events")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", sasl_config)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .option("maxOffsetsPerTrigger", 200000)
    .load()
)

bronze_df = raw_stream.select(
    col("key").cast("string").alias("kafka_key"),
    col("value").cast("string").alias("raw_json"),
    col("topic"), col("partition"), col("offset"),
    col("timestamp").alias("kafka_ts"),
    current_timestamp().alias("ingest_ts"),
)

In [0]:
(
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://checkpoint@adlsstreamingagni.dfs.core.windows.net/bronze_wiki_events/")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("streaming_demo.bronze.wiki_events_brz")
)

In [0]:
# Set table properties once the table exists
spark.sql("""
    ALTER TABLE streaming_demo.bronze.wiki_events_brz
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")